# Session 5 · Part 3 — Sensor Fusion: Why One Sensor Is Not Enough
## 🎓 STUDENT VERSION

**AI for Environmental Sustainability** · Dr. René Traoré, Eva Vormschlag, Prof. Marius Lindauer  
📦 Repository: https://github.com/automl-edu/AI_Environmental_Sustainability

---

## Learning objectives

1. Train separate models on optical (Sentinel-2) and SAR (Sentinel-1) data
2. Implement late-fusion by concatenating backbone features
3. Compare per-class accuracy across all three models

**Estimated time:** ~25 minutes

| Sensor | Modality | What it measures | Limitations |
|--------|----------|-----------------|-------------|
| Sentinel-2 | Optical (13 bands) | Surface reflectance | Cloud cover |
| Sentinel-1 | SAR (VV, VH) | Surface roughness, structure | Speckle noise |

**Dataset:** CopernicusBenchEuroSATS2 (optical) + CopernicusBenchEuroSATS1 (SAR) — torchgeo>=0.6 required  
**Fallback:** So2Sat LCZ42 (set `USE_FALLBACK = True`)


## Section 0 · Environment Setup

In [ ]:
import os, sys

def get_runtime():
    if 'google.colab' in sys.modules:
        return 'colab'
    elif 'SLURM_JOB_ID' in os.environ or 'KISSKI' in os.environ.get('HPC_NAME', ''):
        return 'hpc'
    else:
        return 'local'

RUNTIME = get_runtime()
print(f"Runtime detected: {RUNTIME}")

if RUNTIME == 'colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DATA_ROOT      = '/content/drive/MyDrive/AI4ES_Seminar/data'
        CHECKPOINT_DIR = '/content/drive/MyDrive/AI4ES_Seminar/checkpoints'
        DOWNLOAD       = False
    except Exception as e:
        print(f"Could not mount Google Drive ({e}). Falling back to local storage.")
        DATA_ROOT      = './data'
        CHECKPOINT_DIR = './checkpoints'
        os.makedirs(DATA_ROOT, exist_ok=True)
        os.makedirs(CHECKPOINT_DIR, exist_ok=True)
        DOWNLOAD       = True
elif RUNTIME == 'hpc':
    # On KISSKI, $HOME is on vast-kisski (~120 GB) — use it for data and checkpoints
    # /opt/conda is a separate system partition that is typically full — never write there
    DATA_ROOT      = os.path.join(os.path.expanduser('~'), 'AI4ES_Seminar', 'data')
    CHECKPOINT_DIR = os.path.join(os.path.expanduser('~'), 'AI4ES_Seminar', 'checkpoints')
    os.makedirs(DATA_ROOT,      exist_ok=True)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    DOWNLOAD       = True
else:
    DATA_ROOT      = os.path.expanduser('~/AI4ES_Seminar/data')
    CHECKPOINT_DIR = os.path.expanduser('~/AI4ES_Seminar/checkpoints')
    os.makedirs(DATA_ROOT, exist_ok=True)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    DOWNLOAD       = True

USE_FALLBACK = False   # set True if CopernicusBench download fails
print(f"Dataset mode: {'So2Sat LCZ42 (fallback)' if USE_FALLBACK else 'CopernicusBench EuroSAT'}")


In [ ]:
import subprocess

packages = [
    'torch>=2.0.0', 'torchvision>=0.15.0', 'torchgeo>=0.6.0',
    'numpy>=1.24.0', 'matplotlib>=3.7.0',
    'scikit-learn>=1.3.0', 'seaborn>=0.12.0', 'pandas>=2.0.0'
]

try:
    import torch, torchgeo, sklearn, seaborn, pandas
    print(f'All packages already available (torch {torch.__version__}, torchgeo {torchgeo.__version__})')
except ImportError:
    pip_tmp = os.path.expanduser('~/.local/pip_tmp')
    os.makedirs(pip_tmp, exist_ok=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--user', '--no-cache-dir'] + packages,
        check=True,
        env={**os.environ, 'TMPDIR': pip_tmp}
    )
    print('Packages installed into ~/.local (--user mode)')


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as tv_models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix
import random, time, copy, warnings

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

warnings.filterwarnings("ignore", category=RuntimeWarning, message=".*encountered in matmul")

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
BATCH_SIZE = 64
EPOCHS     = 8
LR         = 1e-3

# macOS/Windows: DataLoader workers can't pickle notebook-defined collate_fn
nw = 2 if sys.platform.startswith('linux') else 0
print(f"Device: {DEVICE}  |  num_workers: {nw}")


## Section 1 · Data Loading

In [ ]:
if not USE_FALLBACK:
    # EuroSATSAR does not exist in TorchGeo — use CopernicusBench equivalents
    # CopernicusBenchEuroSATS2 = Sentinel-2 optical (13 bands)
    # CopernicusBenchEuroSATS1 = Sentinel-1 SAR (VV + VH = 2 channels)
    from torchgeo.datasets import CopernicusBenchEuroSATS1, CopernicusBenchEuroSATS2

    train_opt = CopernicusBenchEuroSATS2(root=DATA_ROOT, split='train', download=True)
    val_opt   = CopernicusBenchEuroSATS2(root=DATA_ROOT, split='val',   download=True)
    test_opt  = CopernicusBenchEuroSATS2(root=DATA_ROOT, split='test',  download=True)
    train_sar = CopernicusBenchEuroSATS1(root=DATA_ROOT, split='train', download=True)
    val_sar   = CopernicusBenchEuroSATS1(root=DATA_ROOT, split='val',   download=True)
    test_sar  = CopernicusBenchEuroSATS1(root=DATA_ROOT, split='test',  download=True)

    OPT_CHANNELS = 13
    SAR_CHANNELS = 2
    NUM_CLASSES  = 10
    CLASS_NAMES  = train_opt.classes
    print(f"Optical: {len(train_opt)}  shape {train_opt[0]['image'].shape}")
    print(f"SAR    : {len(train_sar)}  shape {train_sar[0]['image'].shape}")

else:
    from torchgeo.datasets import So2Sat
    train_ds = So2Sat(root=DATA_ROOT, split='train', download=True)
    val_ds   = So2Sat(root=DATA_ROOT, split='val',   download=True)
    test_ds  = So2Sat(root=DATA_ROOT, split='test',  download=True)
    OPT_CHANNELS = 10; SAR_CHANNELS = 8; NUM_CLASSES = 17
    CLASS_NAMES  = train_ds.classes
    print(f"So2Sat train: {len(train_ds)}")


In [ ]:
def collate_opt(batch):
    return (torch.stack([s['image'].float() for s in batch]),
            torch.tensor([s['label'] for s in batch]))

def collate_sar(batch):
    return (torch.stack([s['image'].float() for s in batch]),
            torch.tensor([s['label'] for s in batch]))

def collate_so2sat_opt(batch):
    return (torch.stack([s['image'][:OPT_CHANNELS].float() for s in batch]),
            torch.tensor([s['label'] for s in batch]))

def collate_so2sat_sar(batch):
    return (torch.stack([s['image'][OPT_CHANNELS:].float() for s in batch]),
            torch.tensor([s['label'] for s in batch]))

if not USE_FALLBACK:
    train_loader_opt = DataLoader(train_opt, BATCH_SIZE, shuffle=True,  collate_fn=collate_opt, num_workers=nw)
    val_loader_opt   = DataLoader(val_opt,   BATCH_SIZE, shuffle=False, collate_fn=collate_opt, num_workers=nw)
    test_loader_opt  = DataLoader(test_opt,  BATCH_SIZE, shuffle=False, collate_fn=collate_opt, num_workers=nw)
    train_loader_sar = DataLoader(train_sar, BATCH_SIZE, shuffle=True,  collate_fn=collate_sar, num_workers=nw)
    val_loader_sar   = DataLoader(val_sar,   BATCH_SIZE, shuffle=False, collate_fn=collate_sar, num_workers=nw)
    test_loader_sar  = DataLoader(test_sar,  BATCH_SIZE, shuffle=False, collate_fn=collate_sar, num_workers=nw)
else:
    train_loader_opt = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  collate_fn=collate_so2sat_opt, num_workers=nw)
    val_loader_opt   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, collate_fn=collate_so2sat_opt, num_workers=nw)
    test_loader_opt  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, collate_fn=collate_so2sat_opt, num_workers=nw)
    train_loader_sar = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  collate_fn=collate_so2sat_sar, num_workers=nw)
    val_loader_sar   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, collate_fn=collate_so2sat_sar, num_workers=nw)
    test_loader_sar  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, collate_fn=collate_so2sat_sar, num_workers=nw)
print("DataLoaders ready.")


## Section 2 · Shared Backbone and Training Infrastructure

In [ ]:
class CNNBackbone(nn.Module):
    def __init__(self, in_channels, feature_dim=512):
        super().__init__()
        base = tv_models.resnet18(weights=None)
        base.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.features    = nn.Sequential(*list(base.children())[:-1])
        self.feature_dim = feature_dim
    def forward(self, x): return self.features(x).flatten(1)

class LinearHead(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_classes))
    def forward(self, x): return self.fc(x)


In [ ]:
def train_one_epoch_single(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total

def train_one_epoch_fusion(model, loader_opt, loader_sar, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for (opt_imgs, labels_o), (sar_imgs, labels_s) in zip(loader_opt, loader_sar):
        assert torch.all(labels_o == labels_s), "Loader alignment error!"
        opt_imgs, sar_imgs = opt_imgs.to(DEVICE), sar_imgs.to(DEVICE)
        labels = labels_o.to(DEVICE)
        optimizer.zero_grad()
        logits = model(opt_imgs, sar_imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * opt_imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += opt_imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate_single(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        preds = model(images.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.tolist()); all_labels.extend(labels.tolist())
    return all_preds, all_labels

@torch.no_grad()
def evaluate_fusion(model, loader_opt, loader_sar):
    model.eval()
    all_preds, all_labels = [], []
    for (opt_imgs, labels_o), (sar_imgs, _) in zip(loader_opt, loader_sar):
        preds = model(opt_imgs.to(DEVICE), sar_imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.tolist()); all_labels.extend(labels_o.tolist())
    return all_preds, all_labels

def run_training_single(model, train_loader, val_loader, epochs, lr=LR, label='model'):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    best_acc, best_state = 0.0, None
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch_single(model, train_loader, optimizer, criterion)
        preds, labels   = evaluate_single(model, val_loader)
        val_acc         = accuracy_score(labels, preds)
        scheduler.step()
        if val_acc > best_acc:
            best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
        print(f"[{label}] Epoch {epoch:02d}/{epochs}  loss={tr_loss:.4f}  "
              f"train={tr_acc:.3f}  val={val_acc:.3f}  ({time.time()-t0:.1f}s)")
    model.load_state_dict(best_state)
    return best_acc, model


## Section 3 · Model A — Optical Only (Sentinel-2)

In [ ]:
class ModelA_Optical(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CNNBackbone(in_channels=OPT_CHANNELS)
        self.head     = LinearHead(self.backbone.feature_dim, NUM_CLASSES)
    def forward(self, x): return self.head(self.backbone(x))

model_a = ModelA_Optical().to(DEVICE)
print(f"Model A parameters: {sum(p.numel() for p in model_a.parameters()):,}")

best_acc_a, model_a = run_training_single(
    model_a, train_loader_opt, val_loader_opt, epochs=EPOCHS, label='ModelA-Optical'
)
print(f"\nModel A (optical) best val accuracy: {best_acc_a:.4f}")


## Section 4 · Model B — SAR Only (Sentinel-1)

In [ ]:
# ── TODO: Model B — SAR only ─────────────────────────────────────────────────
# Mirror ModelA_Optical but change in_channels to SAR_CHANNELS
class ModelB_SAR(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: fill in backbone and head
        raise NotImplementedError
    def forward(self, x):
        raise NotImplementedError

model_b = ModelB_SAR().to(DEVICE)
print(f"Model B parameters: {sum(p.numel() for p in model_b.parameters()):,}")


In [ ]:
# ── TODO: train Model B ───────────────────────────────────────────────────────
# Use run_training_single with train_loader_sar and val_loader_sar
raise NotImplementedError


## Section 5 · Model C — Late Fusion

```
Optical input  →  Backbone A  →  feat_opt [512]  ─┐
                                                    concat [1024]  →  head  →  logits
SAR input      →  Backbone B  →  feat_sar [512]  ─┘
```


In [ ]:
# ── TODO: Model C — Late Fusion ──────────────────────────────────────────────
# 1. Add backbone_opt (OPT_CHANNELS) and backbone_sar (SAR_CHANNELS)
# 2. Compute fused_dim = backbone_opt.feature_dim + backbone_sar.feature_dim
# 3. Add a LinearHead on fused_dim
# 4. In forward: extract features from both, concatenate, pass through head
class ModelC_LateFusion(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO
        raise NotImplementedError
    def forward(self, x_opt, x_sar):
        # TODO
        raise NotImplementedError


In [ ]:
model_c = ModelC_LateFusion().to(DEVICE)
print(f"Model C parameters: {sum(p.numel() for p in model_c.parameters()):,}")

# Fusion training — loaders must be unshuffled so indices align across modalities
train_loader_opt_ns = DataLoader(
    train_opt if not USE_FALLBACK else train_ds,
    BATCH_SIZE, shuffle=False,
    collate_fn=collate_opt if not USE_FALLBACK else collate_so2sat_opt, num_workers=nw
)
train_loader_sar_ns = DataLoader(
    train_sar if not USE_FALLBACK else train_ds,
    BATCH_SIZE, shuffle=False,
    collate_fn=collate_sar if not USE_FALLBACK else collate_so2sat_sar, num_workers=nw
)

optimizer_c = optim.Adam(model_c.parameters(), lr=LR)
scheduler_c = optim.lr_scheduler.CosineAnnealingLR(optimizer_c, T_max=EPOCHS)
criterion   = nn.CrossEntropyLoss()
best_acc_c, best_state_c = 0.0, None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch_fusion(
        model_c, train_loader_opt_ns, train_loader_sar_ns, optimizer_c, criterion)
    preds, labels = evaluate_fusion(model_c, val_loader_opt, val_loader_sar)
    val_acc       = accuracy_score(labels, preds)
    scheduler_c.step()
    if val_acc > best_acc_c:
        best_acc_c, best_state_c = val_acc, copy.deepcopy(model_c.state_dict())
    print(f"[ModelC-Fusion] Epoch {epoch:02d}/{EPOCHS}  loss={tr_loss:.4f}  "
          f"train={tr_acc:.3f}  val={val_acc:.3f}  ({time.time()-t0:.1f}s)")

model_c.load_state_dict(best_state_c)
print(f"\nModel C (fusion) best val accuracy: {best_acc_c:.4f}")


## Section 6 · Per-class Accuracy Comparison

In [ ]:
def per_class_accuracy(preds, labels):
    cm   = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    with np.errstate(divide='ignore', invalid='ignore'):
        accs = np.diag(cm) / cm.sum(axis=1)
    return np.nan_to_num(accs)


In [ ]:
# ── TODO: evaluate all three models on the test set ──────────────────────────
# Use evaluate_single for A and B, evaluate_fusion for C
raise NotImplementedError


In [ ]:
acc_a = per_class_accuracy(preds_a, labels_a)
acc_b = per_class_accuracy(preds_b, labels_b)
acc_c = per_class_accuracy(preds_c, labels_c)
oa_a  = accuracy_score(labels_a, preds_a)
oa_b  = accuracy_score(labels_b, preds_b)
oa_c  = accuracy_score(labels_c, preds_c)
print(f"Overall accuracy — Optical: {oa_a:.3f}  SAR: {oa_b:.3f}  Fusion: {oa_c:.3f}")

fig, ax = plt.subplots(figsize=(14, 5))
x, width = np.arange(NUM_CLASSES), 0.25
bars_a = ax.bar(x - width, acc_a, width, label=f'Optical (OA={oa_a:.3f})', color='steelblue',  alpha=0.85)
bars_b = ax.bar(x,          acc_b, width, label=f'SAR     (OA={oa_b:.3f})', color='darkorange', alpha=0.85)
bars_c = ax.bar(x + width,  acc_c, width, label=f'Fusion  (OA={oa_c:.3f})', color='seagreen',   alpha=0.85)

for bars in [bars_a, bars_b, bars_c]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.05:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}',
                    ha='center', va='bottom', fontsize=6, rotation=90)

ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=40, ha='right', fontsize=9)
ax.set_ylim(0, 1.15); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_title('Per-class accuracy: Optical vs SAR vs Late Fusion')
plt.tight_layout(); plt.savefig('sensor_fusion_per_class_accuracy.png', dpi=150); plt.show()


In [ ]:
best_modality = []
for a, b, c in zip(acc_a, acc_b, acc_c):
    best = max(a, b, c)
    best_modality.append('Fusion' if c == best else 'Optical' if a == best else 'SAR')

summary = pd.DataFrame({
    'Class': CLASS_NAMES,
    'Optical (%)': (acc_a * 100).round(1),
    'SAR (%)':     (acc_b * 100).round(1),
    'Fusion (%)':  (acc_c * 100).round(1),
    'Fusion vs Opt Δ': ((acc_c - acc_a) * 100).round(1),
    'Best': best_modality
})
print(summary.to_string(index=False))
print(f"\nFusion wins: {(summary['Best']=='Fusion').sum()}  "
      f"Optical: {(summary['Best']=='Optical').sum()}  "
      f"SAR: {(summary['Best']=='SAR').sum()}")


## Discussion

1. Which classes does SAR outperform optical? Explain physically.
2. Which classes clearly favour optical? Is this consistent with spectral distinctiveness?
3. Does fusion ever underperform both single-modality models on a specific class?
4. What would early fusion (concatenate raw bands) gain or lose vs late fusion?
5. How does this connect to the biodiversity monitoring paper in Session 6?

